# Level 2 프로젝트 스타터: Autonomous Vision Agent

과제:

> Find and sort the six cubes in the warehouse into their matching destination pads.

Level 2에서는 scene_state, 정확한 entity ID, coordinate go_to를 사용할 수 없습니다. camera observation, set_head, set_velocity, memory, recovery로 이동합니다.

실행 전에 `MENLO_API_KEY`와 `TOKAMAK_API_KEY`를 모두 설정하세요. 프로젝트는 text LLM decision loop를 필수로 요구합니다.


## 사용 방법

아래 코드는 완성된 solution이 아니라 최소 scaffold입니다. SUPPORT CODE는 setup, wrapper, schema, memory 구조를 제공합니다. STUDENT TODO 영역에서 LLM decision, observation, action execution, verification, memory update 전략을 직접 설계하세요.


In [ ]:
# Colab/로컬 setup. 필요하면 한 번 실행하세요.
%pip install -q "git+https://github.com/asimovinc/hansung-menlo-robotics-workshop.git" python-dotenv opencv-python Pillow matplotlib


## 스타터 Scaffold

이 cell을 실행한 뒤 TODO 영역을 팀 설계에 맞게 수정하세요.


In [ ]:
"""Level 2 프로젝트 스타터입니다.

이 파일은 완성된 해답이 아니라 최소 scaffold입니다.

SUPPORT CODE 영역은 반복해서 작성할 필요가 없는 wrapper, 자료 구조,
schema validation을 제공합니다. STUDENT TODO 영역은 팀이 직접 설계하고,
개선하고, 테스트하고, 발표에서 설명해야 하는 부분입니다.

Level 2 규칙: `scene_state`, 정확한 entity ID, coordinate `go_to`를 사용할 수
없습니다. 카메라 관찰값, `set_head`, `set_velocity`, memory, recovery로
navigation을 구현해야 합니다.
"""

import asyncio
import json
import math
from dataclasses import dataclass, field
from typing import Any

from menlo_runner.llm import ask_vlm
from menlo_runner.perception import detect_color_blobs


# ---------------------------------------------------------------------------
# SUPPORT CODE: 공통 과제 정의와 필수 LLM decision schema
# ---------------------------------------------------------------------------
TASK = "Find and sort the six cubes in the warehouse into their matching destination pads."

DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}
SIGNAGE_NOTE = (
    "A is the conveyor/cube source area, not a destination. "
    "Destination signs are B red, C green, D blue, E yellow."
)

ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


@dataclass
class AgentDecision:
    """LLM이 반환하고 코드가 검증한 고수준 decision입니다."""

    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    """observe-decide-act cycle 사이에 유지하는 agent 상태입니다."""

    delivered_count: int = 0
    delivery_limit: int | None = None
    priority_colors: list[str] = field(default_factory=list)
    held_color: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    search_turns: int = 0
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_colors: list[str] = field(default_factory=list)
    skipped_colors: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    """LLM과 action code에 전달할 compact observation입니다."""

    robot_status: Any
    detections: list[Any]
    note: str = ""
    vlm_summary: str = ""


@dataclass(frozen=True)
class ScannedDetection:
    """head pose를 함께 저장한 색상 detection입니다."""

    color: str
    angle_deg: float
    blob_area: int
    centroid: tuple[int, int]
    bbox: tuple[int, int, int, int]
    head_yaw: float
    head_pitch: float

    @property
    def full_bearing_deg(self) -> float:
        """image angle에 head yaw를 더한 대략적인 body-relative bearing입니다."""
        return self.angle_deg + math.degrees(self.head_yaw)


def parse_agent_decision(text: str) -> AgentDecision | None:
    """LLM의 JSON 응답을 parse하고 필수 schema를 검증합니다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]
    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """robot state를 LLM에 전달하기 좋은 compact text context로 변환합니다."""
    visible = [
        {
            "color": detection.color,
            "angle_deg": detection.angle_deg,
            "full_bearing_deg": round(getattr(detection, "full_bearing_deg", detection.angle_deg), 1),
            "blob_area": detection.blob_area,
            "bbox": detection.bbox,
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "active_color": memory.active_color,
        "stage": memory.stage,
        "delivered_count": memory.delivered_count,
        "delivery_limit": memory.delivery_limit,
        "priority_colors": memory.priority_colors,
        "completed_colors": memory.completed_colors,
        "skipped_colors": memory.skipped_colors,
        "failed_attempts": memory.failed_attempts,
        "last_result": last_result,
        "note": observation.note,
        "signage_note": SIGNAGE_NOTE,
        "vlm_summary": observation.vlm_summary,
    }


# ---------------------------------------------------------------------------
# SUPPORT CODE: Level 2에서 허용되는 SDK wrapper
# ---------------------------------------------------------------------------
# 이 wrapper에는 scene_state, 정답 좌표, 정확한 cube/pad entity ID, go_to를
# 추가하지 마세요.

async def get_robot_status(ctx: Any) -> Any:
    """robot pose, motion status, neck state를 읽습니다."""
    return await ctx.state("robot_status")


async def get_camera_frame(ctx: Any) -> bytes:
    """POV camera frame을 가져옵니다."""
    return await ctx.get_vision("pov")


def build_signage_vlm_prompt(held_color: str | None = None) -> str:
    """고정 창고 표지판을 읽기 위한 VLM prompt입니다."""
    target = ""
    if held_color in DESTINATION_SIGN_RULES:
        target = f" The robot is holding a {held_color} cube, so the target destination sign is {DESTINATION_SIGN_RULES[held_color]}."
    return (
        "Read the floating warehouse signs visible in this robot camera frame. "
        f"{SIGNAGE_NOTE} "
        "Return JSON with visible sign letters, colors, rough left/center/right positions, and confidence."
        + target
    )


async def ask_vlm_about_frame(ctx: Any, prompt: str, *, api_key: str) -> str:
    """현재 POV frame에 대해 project-allowed VLM helper로 질문합니다."""
    jpeg = await get_camera_frame(ctx)
    return ask_vlm(jpeg, prompt, api_key=api_key)


async def perceive(ctx: Any) -> list[Any]:
    """현재 camera frame에서 Workshop 2 color-blob detector를 실행합니다."""
    jpeg = await get_camera_frame(ctx)
    return detect_color_blobs(jpeg)


async def set_head(ctx: Any, *, yaw: float | None = None, pitch: float | None = None) -> Any:
    """walking direction은 바꾸지 않고 camera 방향만 조정합니다."""
    args: dict[str, float] = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    return await ctx.invoke("set_head", args, timeout_s=10)


async def move_velocity(
    ctx: Any,
    *,
    vx: float = 0.0,
    vy: float = 0.0,
    wz: float = 0.0,
    duration_s: float = 1.0,
) -> Any:
    """짧은 body-frame velocity command를 보냅니다."""
    return await ctx.invoke(
        "set_velocity",
        {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
        timeout_s=30,
    )


async def cancel_action(ctx: Any) -> Any:
    """현재 실행 중인 runtime action을 취소합니다."""
    return await ctx.invoke("cancel", {})


async def pick_nearest_cube(ctx: Any) -> Any:
    """의도한 cube 가까이 시각적으로 이동한 뒤 nearest cube를 집습니다."""
    return await ctx.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": "cube"}},
        timeout_s=300,
    )


async def place_nearest_zone(ctx: Any) -> Any:
    """matching pad 가까이 이동한 뒤 nearest zone에 내려놓습니다."""
    return await ctx.invoke("place_entity", {}, timeout_s=300)


def result_summary(result: Any) -> dict[str, Any]:
    """SDK result를 log에 넣기 쉬운 작은 dictionary로 변환합니다."""
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    return {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }


async def scan_head(
    ctx: Any,
    *,
    yaws: tuple[float, ...] = (-0.8, 0.0, 0.8),
    pitch: float = 0.15,
) -> list[Any]:
    """간단한 scan helper입니다. 더 나은 search 전략으로 교체할 수 있습니다."""
    all_detections: list[Any] = []
    for yaw in yaws:
        await set_head(ctx, yaw=yaw, pitch=pitch)
        await asyncio.sleep(0.4)
        for detection in await perceive(ctx):
            all_detections.append(
                ScannedDetection(
                    color=detection.color,
                    angle_deg=detection.angle_deg,
                    blob_area=detection.blob_area,
                    centroid=detection.centroid,
                    bbox=detection.bbox,
                    head_yaw=yaw,
                    head_pitch=pitch,
                )
            )
    return all_detections


# ---------------------------------------------------------------------------
# STUDENT TODO: LLM decision 함수
# ---------------------------------------------------------------------------

async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """text LLM으로 다음 고수준 action을 선택합니다.

    LLM은 high-level action만 고르고, 속도/좌표/id 제어는 아래 deterministic
    execution 함수가 camera feedback으로 처리합니다. LLM 호출이나 JSON 검증이
    실패하면 stage-aware 안전 fallback으로 recover/search를 선택합니다.
    """
    decision_context = build_decision_context(task, observation, memory, last_result)

    allowed = sorted(ALLOWED_NEXT_ACTIONS)
    prompt = (
        "You are the high-level planner for a Level 2 autonomous vision robot. "
        "Return ONLY one JSON object with keys next_action, target_color, reason, "
        "and optional recovery_strategy. Do not output velocity, coordinates, scene_state, "
        "entity ids, or low-level commands. Valid next_action values are: "
        f"{allowed}. Use camera detections, memory stage, and last_result only. "
        "If holding a cube, search/navigate/place only the matching destination color. "
        "If not holding a cube, search/navigate/pick a visible cube color."
    )
    user = json.dumps(decision_context, ensure_ascii=False, default=str)

    try:
        from menlo_runner.llm import call_llm

        reply = call_llm([
            {"role": "system", "content": prompt},
            {"role": "user", "content": user},
        ])
        parsed = parse_agent_decision(str(reply))
        if parsed is not None:
            return parsed
    except Exception as exc:
        decision_context["llm_error"] = str(exc)[:200]

    # 안전 fallback: LLM JSON/action 오류 또는 helper 미설정 시에도 schema 내부 action만 사용합니다.
    visible = decision_context["visible_targets"]
    failed_total = sum(memory.failed_attempts.values())
    if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
        return AgentDecision(next_action="stop", reason="Fallback: delivery limit reached.")
    if failed_total >= 10:
        return AgentDecision(next_action="recover", reason="Fallback: repeated failures.", recovery_strategy="wide_rescan")

    if memory.held_color:
        target_color = memory.held_color
        if last_result and last_result.get("ok") and last_result.get("action") == "navigate_to_pad":
            return AgentDecision(next_action="place_cube", target_color=target_color, reason="Fallback: matching pad reached.")
        if any(item["color"] == target_color for item in visible):
            return AgentDecision(next_action="navigate_to_pad", target_color=target_color, reason="Fallback: matching destination color visible.")
        return AgentDecision(next_action="search_pad", target_color=target_color, reason="Fallback: holding cube, need matching pad.")

    if last_result and last_result.get("ok") and last_result.get("action") == "navigate_to_cube":
        return AgentDecision(next_action="pick_cube", target_color=memory.active_color, reason="Fallback: cube reached.")

    candidates = [item for item in visible if item["color"] not in memory.completed_colors]
    if memory.priority_colors:
        candidates.sort(key=lambda item: (item["color"] not in memory.priority_colors, -item["blob_area"]))
    if candidates:
        largest = max(candidates, key=lambda item: item["blob_area"])
        return AgentDecision(
            next_action="navigate_to_cube",
            target_color=largest["color"],
            reason="Fallback: choose largest visible unfinished color blob.",
        )

    return AgentDecision(next_action="search_cube", reason="Fallback: no visible unfinished target.")


# ---------------------------------------------------------------------------
# STUDENT TODO: observation, verification, memory
# ---------------------------------------------------------------------------

async def observe_world(ctx: Any, memory: AgentMemory) -> Observation:
    """LLM과 action code에 전달할 현재 observation을 수집합니다.

    stage에 따라 full head scan과 중앙 frame 관찰을 섞습니다. scene_state나
    entity id는 사용하지 않습니다.
    """
    robot_status = await get_robot_status(ctx)
    if memory.stage in {"need_cube", "search_cube", "need_pad", "search_pad"} or memory.search_turns % 3 == 0:
        detections = await scan_head(ctx)
        note = "full_head_scan"
    else:
        await set_head(ctx, yaw=0.0, pitch=0.12)
        detections = [
            ScannedDetection(
                color=d.color,
                angle_deg=d.angle_deg,
                blob_area=d.blob_area,
                centroid=d.centroid,
                bbox=d.bbox,
                head_yaw=0.0,
                head_pitch=0.12,
            )
            for d in await perceive(ctx)
        ]
        note = "center_frame"
    return Observation(robot_status=robot_status, detections=detections, note=note)


async def verify_outcome(ctx: Any, decision: AgentDecision, action_result: dict[str, Any]) -> dict[str, Any]:
    """마지막 action이 성공했는지 camera evidence와 SDK result로 확인합니다."""
    action = action_result.get("action", decision.next_action)
    sdk_status = action_result.get("result", {}).get("status")
    ok = False
    evidence: dict[str, Any] = {"sdk_status": sdk_status}

    if action in {"search_cube", "search_pad"}:
        ok = bool(action_result.get("found"))
    elif action in {"navigate_to_cube", "navigate_to_pad"}:
        ok = bool(action_result.get("reached"))
    elif action == "pick_cube":
        ok = sdk_status == "done"
        evidence["held_color_hint"] = decision.target_color
    elif action == "place_cube":
        ok = sdk_status == "done"
        evidence["placed_color_hint"] = decision.target_color
    elif action == "recover":
        ok = action_result.get("status") in {"recovered", "stepped_back_and_rotated"}
    elif action in {"skip_target", "stop"}:
        ok = True

    # 다음 cycle의 LLM/recovery가 볼 수 있도록 작은 재관찰 evidence를 추가합니다.
    try:
        detections = await scan_head(ctx, yaws=(0.0,), pitch=0.12)
        evidence["post_visible"] = [
            {"color": d.color, "bearing": round(d.full_bearing_deg, 1), "area": d.blob_area}
            for d in detections[:6]
        ]
    except Exception as exc:
        evidence["post_observe_error"] = str(exc)[:160]

    return {
        "ok": ok,
        "action": action,
        "target_color": decision.target_color,
        "decision": decision.__dict__,
        "action_result": action_result,
        "evidence": evidence,
    }


def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """각 cycle 뒤 persistent state를 갱신합니다."""
    action = verified.get("action", decision.next_action)
    ok = bool(verified.get("ok"))
    color = decision.target_color or memory.active_color or memory.held_color

    if decision.next_action in {"search_cube", "search_pad"}:
        memory.search_turns += 1
    else:
        memory.search_turns = 0

    if decision.next_action in {"navigate_to_cube", "search_cube"} and decision.target_color:
        memory.active_color = decision.target_color
        memory.stage = "approach_cube" if decision.next_action == "navigate_to_cube" else "search_cube"

    if action == "navigate_to_cube" and ok:
        memory.stage = "ready_to_pick"
    elif action == "pick_cube" and ok:
        memory.held_color = decision.target_color or memory.active_color
        memory.active_color = memory.held_color
        memory.stage = "need_pad"
    elif decision.next_action in {"search_pad", "navigate_to_pad"}:
        memory.stage = "approach_pad" if decision.next_action == "navigate_to_pad" else "search_pad"
    elif action == "navigate_to_pad" and ok:
        memory.stage = "ready_to_place"
    elif action == "place_cube" and ok:
        placed_color = decision.target_color or memory.held_color
        if placed_color and placed_color not in memory.completed_colors:
            memory.completed_colors.append(placed_color)
        memory.delivered_count += 1
        memory.held_color = None
        memory.active_color = None
        memory.stage = "need_cube"
    elif decision.next_action == "skip_target":
        if color and color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        memory.active_color = None
        if memory.held_color is None:
            memory.stage = "need_cube"
    elif decision.next_action == "recover":
        memory.stage = "need_pad" if memory.held_color else "need_cube"

    if ok:
        if color:
            memory.failed_attempts[color] = 0
    else:
        key = color or decision.next_action
        memory.failed_attempts[key] = memory.failed_attempts.get(key, 0) + 1
        if memory.failed_attempts[key] >= 3 and not memory.held_color and key not in memory.skipped_colors:
            memory.skipped_colors.append(key)
            memory.active_color = None
            memory.stage = "need_cube"

    memory.logs.append(
        {
            "observation": {
                "visible_count": len(observation.detections),
                "note": observation.note,
                "colors": sorted({d.color for d in observation.detections}),
            },
            "llm_decision": decision.__dict__,
            "verified": verified,
            "memory": {
                "stage": memory.stage,
                "active_color": memory.active_color,
                "held_color": memory.held_color,
                "delivered_count": memory.delivered_count,
                "completed_colors": list(memory.completed_colors),
                "failed_attempts": dict(memory.failed_attempts),
            },
        }
    )


# ---------------------------------------------------------------------------
# LEVEL 2 STUDENT TODO: vision-only action 구현
# ---------------------------------------------------------------------------
# Level 2에서는 go_to를 호출하면 안 됩니다. camera observation, set_head,
# set_velocity, memory, recovery behavior로 navigation을 구현하세요.

def _best_detection(detections: list[Any], target_color: str | None = None) -> Any | None:
    candidates = [d for d in detections if target_color is None or d.color == target_color]
    if not candidates:
        return None
    return max(candidates, key=lambda d: d.blob_area)


async def visual_search(ctx: Any, target_color: str | None = None) -> bool:
    """camera movement와 robot motion으로 cube 또는 pad를 찾습니다."""
    scan_patterns = [
        (-0.9, -0.45, 0.0, 0.45, 0.9),
        (-0.6, 0.0, 0.6),
    ]
    for yaws in scan_patterns:
        detections = await scan_head(ctx, yaws=tuple(yaws), pitch=0.12)
        if _best_detection(detections, target_color) is not None:
            return True
        await move_velocity(ctx, wz=0.45, duration_s=0.7)
    await set_head(ctx, yaw=0.0, pitch=0.12)
    return False


async def visual_navigate_to_target(ctx: Any, target_color: str | None) -> bool:
    """선택한 target까지 closed-loop vision-only navigation을 수행합니다."""
    close_area = 4200
    centered_deg = 8.0

    await set_head(ctx, yaw=0.0, pitch=0.10)
    for _ in range(12):
        detections = await scan_head(ctx, yaws=(0.0,), pitch=0.10)
        best = _best_detection(detections, target_color)
        if best is None:
            detections = await scan_head(ctx, yaws=(-0.55, 0.0, 0.55), pitch=0.12)
            best = _best_detection(detections, target_color)
        if best is None:
            await move_velocity(ctx, wz=0.35, duration_s=0.6)
            continue

        bearing = float(getattr(best, "full_bearing_deg", best.angle_deg))
        area = int(best.blob_area)
        if abs(bearing) <= centered_deg and area >= close_area:
            await set_head(ctx, yaw=0.0, pitch=0.10)
            return True

        # angle_deg > 0이면 target이 오른쪽에 있으므로 일반적으로 wz < 0.
        turn = max(-0.45, min(0.45, -0.018 * bearing))
        forward = 0.0 if abs(bearing) > 22 else (0.10 if area < close_area else 0.04)
        await move_velocity(ctx, vx=forward, wz=turn, duration_s=0.55)

    # 마지막에 가까운 조작 가능 상태인지 한 번 더 확인합니다.
    detections = await scan_head(ctx, yaws=(0.0,), pitch=0.10)
    best = _best_detection(detections, target_color)
    return bool(best and abs(getattr(best, "full_bearing_deg", best.angle_deg)) <= 12 and best.blob_area >= close_area * 0.75)


async def recover_motion(ctx: Any, memory: AgentMemory, reason: str | None = None) -> dict[str, Any]:
    """target loss, blocked motion, failed manipulation에서 회복합니다."""
    failures = sum(memory.failed_attempts.values())
    await set_head(ctx, yaw=0.0, pitch=0.12)
    await move_velocity(ctx, vx=-0.15, duration_s=0.8)
    if failures >= 4 or reason == "wide_rescan":
        await move_velocity(ctx, wz=0.45, duration_s=1.4)
        found = await visual_search(ctx, memory.held_color or memory.active_color)
        return {"action": "recover", "reason": reason, "status": "recovered", "wide_rescan": True, "found": found}
    await move_velocity(ctx, wz=0.35, duration_s=0.8)
    return {"action": "recover", "reason": reason, "status": "recovered", "wide_rescan": False}


async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """검증된 LLM decision 하나를 Level 2 robot action으로 변환합니다."""
    if decision.next_action in {"search_cube", "search_pad"}:
        found = await visual_search(ctx, decision.target_color or memory.held_color)
        return {"action": decision.next_action, "found": found}

    if decision.next_action in {"navigate_to_cube", "navigate_to_pad"}:
        reached = await visual_navigate_to_target(ctx, decision.target_color or memory.held_color)
        return {"action": decision.next_action, "reached": reached}

    if decision.next_action == "pick_cube":
        result = await pick_nearest_cube(ctx)
        return {"action": "pick_cube", "result": result_summary(result)}

    if decision.next_action == "place_cube":
        result = await place_nearest_zone(ctx)
        return {"action": "place_cube", "result": result_summary(result)}

    if decision.next_action == "recover":
        return await recover_motion(ctx, memory, decision.recovery_strategy)

    if decision.next_action == "skip_target":
        return {"action": "skip_target", "status": "skipped", "target_color": decision.target_color}

    if decision.next_action == "stop":
        return {"action": "stop", "status": "stopped"}

    return {"action": decision.next_action, "status": "no_op"}


async def run_agent(ctx: Any, *, max_cycles: int = 36) -> AgentMemory:
    """얇은 observe-LLM-act loop입니다. loop만이 아니라 TODO 함수들을 수정하세요."""
    memory = AgentMemory()
    if memory.delivery_limit is None:
        memory.delivery_limit = 6
    last_result: dict[str, Any] | None = None

    for cycle in range(1, max_cycles + 1):
        print(f"\n[Level 2] Cycle {cycle}")
        if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
            memory.stage = "done"
            break

        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(TASK, observation, memory, last_result)
        print("LLM decision:", decision)

        if decision.next_action == "stop":
            memory.stage = "done"
            break

        action_result = await execute_decision(ctx, decision, observation, memory)
        verified = await verify_outcome(ctx, decision, action_result)
        update_memory(memory, observation, decision, verified)
        last_result = verified

        if not verified.get("ok") and sum(memory.failed_attempts.values()) >= 12:
            print("Stopping after repeated failures; inspect memory.logs for recovery details.")
            break

    return memory


async def run(ctx: Any) -> None:
    print(TASK)
    print("Running Level 2 autonomous-vision project starter")
    memory = await run_agent(ctx)
    print("\nRun complete.")
    print(f"Delivered count: {memory.delivered_count}")
    print("Logs:")
    for item in memory.logs:
        print(item)



## 로봇 연결

Prompt가 나오면 viewer URL을 Chrome에서 여세요.


In [ ]:
from menlo_runner.config import load_config
from menlo_runner.context import open_robot_context

config = load_config(require_tokamak=True)
ctx = await open_robot_context(config, name_prefix="level-2-notebook-ko")
print(ctx.viewer_url)


## Agent 실행

현재 실험에 필요한 TODO를 충분히 구현한 뒤 실행하세요.


In [ ]:
memory = await run_agent(ctx)
memory.logs


In [ ]:
# 종료할 때 cleanup하세요.
# await ctx.close()
